# Cleaning IVV wine production data

This notebook cleans the file `36_Evolu__o_da_Produ__o_Nacional_por_Reg7.xls` and creates a machine-learning-ready CSV in `data/processed`.

Expected input:

```text
data/raw/36_Evolu__o_da_Produ__o_Nacional_por_Reg7.xls
```

Expected output:

```text
data/processed/wine_production_by_region_clean.csv
```

The notebook is designed for the IVV Excel structure where each table has regions in rows and campaign years in columns, with percentage columns intercalated. It extracts only production volume columns in hectolitres and ignores percentage columns and `Total Geral`.


In [1]:

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)


In [2]:

NOTEBOOK_DIR = Path.cwd()

# If this notebook is opened from the notebooks folder, ROOT_DIR should be the parent folder.
# If it is opened from the repository root, ROOT_DIR stays as the current folder.
if NOTEBOOK_DIR.name == "notebooks":
    ROOT_DIR = NOTEBOOK_DIR.parent
else:
    ROOT_DIR = NOTEBOOK_DIR

RAW_FILE = ROOT_DIR / "data" / "raw" / "36_Evolu__o_da_Produ__o_Nacional_por_Reg7.xls"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
OUTPUT_FILE = PROCESSED_DIR / "wine_production_by_region_clean.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Root directory:", ROOT_DIR)
print("Raw file:", RAW_FILE)
print("Output file:", OUTPUT_FILE)
print("Raw file exists?", RAW_FILE.exists())


Root directory: c:\Users\islec\pml_final_project
Raw file: c:\Users\islec\pml_final_project\data\raw\36_Evolu__o_da_Produ__o_Nacional_por_Reg7.xls
Output file: c:\Users\islec\pml_final_project\data\processed\wine_production_by_region_clean.csv
Raw file exists? True


## 1. Load all sheets

The file may contain several sheets or several tables inside the same sheet. Therefore, the notebook reads every sheet without assuming a fixed header row.


In [3]:

    """Load an Excel file as a dictionary of raw DataFrames.

    Some public-sector .xls files are real Excel files, while others are HTML tables
    saved with an .xls extension. This function handles both cases.
    """
    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path}\n"
            "Check if the file is inside data/raw and if the filename is correct."
        )

    try:
        sheets = pd.read_excel(file_path, sheet_name=None, header=None, dtype=str)
        print(f"Loaded {len(sheets)} Excel sheet(s).")
        return sheets
    except Exception as excel_error:
        print("pd.read_excel failed. Trying pd.read_html...")
        print("Excel error:", excel_error)

        html_tables = pd.read_html(file_path, header=None)
        sheets = {f"html_table_{i+1}": table.astype(str) for i, table in enumerate(html_tables)}
        print(f"Loaded {len(sheets)} HTML table(s).")
        return sheets


raw_sheets = load_excel_or_html_tables(RAW_FILE)

for sheet_name, df in raw_sheets.items():
    print(f"{sheet_name}: {df.shape}")


IndentationError: unexpected indent (3755189017.py, line 1)

## 2. Helper functions

These functions identify the table type, clean Portuguese number formats, locate campaign-year columns, and transform each table into long format.


In [ ]:

CAMPAIGN_PATTERN = re.compile(r"^20\d{2}/\d{2}$")

TYPE_MAP = {
    "total": "total_production_hl",
    "dop": "dop_production_hl",
    "igp": "igp_production_hl",
    "ano_casta": "year_variety_production_hl",
    "sem_do_ig": "non_certified_production_hl",
}

REGIONS_TO_KEEP = {
    "Verdes",
    "T. Montes",
    "Douro",
    "Bairrada",
    "Dão",
    "Beira Interior",
    "Távora-Varosa",
    "Tejo",
    "Lisboa",
    "P. Setúbal",
    "Alentejo",
    "Algarve",
    "Madeira",
    "Açores",
}


def normalize_text(value):
    \"\"\"Convert a cell value to a stripped string.\"\"\"
    if pd.isna(value):
        return ""
    value = str(value).replace("\\xa0", " ").strip()
    value = re.sub(r"\\s+", " ", value)
    return value


def clean_number_pt(value):
    \"\"\"Convert Portuguese-formatted numbers such as '1.004.727' or '16,9' to float.

    For production volume columns, the expected values are mostly integers.
    Percentage columns are ignored elsewhere.
    \"\"\"
    text = normalize_text(value)

    if text == "" or text.lower() in {"nan", "none"}:
        return np.nan

    # Keep only digits, minus sign, dots and commas.
    text = re.sub(r"[^0-9,\\.\\-]", "", text)

    if text == "":
        return np.nan

    # Portuguese format: dot as thousands separator and comma as decimal separator.
    text = text.replace(".", "").replace(",", ".")

    try:
        return float(text)
    except ValueError:
        return np.nan


def infer_table_type(title_text):
    \"\"\"Infer which production type a table represents based on its title.\"\"\"
    title = normalize_text(title_text).lower()

    if "total por região" in title or "total por regiao" in title:
        return "total"

    if "denominação de origem protegida" in title or "denominacao de origem protegida" in title or "dop" in title:
        return "dop"

    if "indicação geográfica protegida" in title or "indicacao geografica protegida" in title or "igp" in title:
        return "igp"

    if "ano/casta" in title or "ano / casta" in title:
        return "ano_casta"

    if "sem do/ig" in title or "sem do / ig" in title:
        return "sem_do_ig"

    return None


def find_title_above(df, header_row, max_rows_above=8):
    \"\"\"Find the closest title above a detected header row.\"\"\"
    start = max(0, header_row - max_rows_above)
    candidate_rows = []

    for r in range(start, header_row):
        row_text = " ".join(normalize_text(x) for x in df.iloc[r].tolist())
        if "Evolução" in row_text or "Evolucao" in row_text:
            candidate_rows.append(row_text)

    if candidate_rows:
        return candidate_rows[-1]

    return ""


def detect_header_rows(df):
    \"\"\"Detect header rows that contain campaign years such as 2025/26.\"\"\"
    header_rows = []

    for idx in range(len(df)):
        row_values = [normalize_text(x) for x in df.iloc[idx].tolist()]
        campaign_count = sum(bool(CAMPAIGN_PATTERN.match(x)) for x in row_values)

        # A real header row should have several campaign-year columns.
        if campaign_count >= 3:
            header_rows.append(idx)

    return header_rows


def parse_table_from_header(df, sheet_name, header_row):
    \"\"\"Parse one IVV table starting from a detected header row.\"\"\"
    title = find_title_above(df, header_row)
    table_type = infer_table_type(title)

    if table_type is None:
        print(f"Skipping table in sheet '{sheet_name}', row {header_row}: could not infer table type.")
        print("Title found:", title[:120])
        return None

    header_values = [normalize_text(x) for x in df.iloc[header_row].tolist()]

    # Find columns whose header is a campaign year.
    year_columns = {
        col_idx: value
        for col_idx, value in enumerate(header_values)
        if CAMPAIGN_PATTERN.match(value)
    }

    if not year_columns:
        return None

    # The region column is usually the first column before the first year column.
    first_year_col = min(year_columns.keys())
    region_col = max(0, first_year_col - 1)

    records = []

    for r in range(header_row + 1, len(df)):
        region = normalize_text(df.iat[r, region_col])

        # Stop if the table reached the total row.
        if region.lower() in {"total geral", "total"}:
            break

        # Skip blank or invalid region rows.
        if region == "" or region.lower() in {"nan", "região vitivinícola", "regiao vitivinicola"}:
            continue

        # Keep known region names only. This prevents notes or extra text from entering the dataset.
        if region not in REGIONS_TO_KEEP:
            continue

        for c, campaign_year in year_columns.items():
            value = clean_number_pt(df.iat[r, c])

            records.append({
                "sheet_name": sheet_name,
                "production_type": table_type,
                "production_column": TYPE_MAP[table_type],
                "region": region,
                "campaign_year": campaign_year,
                "year_start": int(campaign_year[:4]),
                "production_hl": value,
            })

    if not records:
        return None

    parsed = pd.DataFrame(records)
    return parsed


## 3. Parse all production tables

This section extracts the volume columns from each table and combines them into one long dataframe.


In [ ]:

all_long_tables = []

for sheet_name, df in raw_sheets.items():
    header_rows = detect_header_rows(df)
    print(f"Sheet/table '{sheet_name}' - detected header rows: {header_rows}")

    for header_row in header_rows:
        parsed = parse_table_from_header(df, sheet_name, header_row)

        if parsed is not None:
            all_long_tables.append(parsed)
            table_type = parsed["production_type"].iloc[0]
            print(f"  Parsed {table_type}: {parsed.shape[0]} rows")

if not all_long_tables:
    raise ValueError(
        "No valid production tables were parsed. "
        "Open the raw file and check if the header rows contain campaign years like 2025/26."
    )

production_long = pd.concat(all_long_tables, ignore_index=True)

display(production_long.head())
print(production_long.shape)

summary = (
    production_long
    .groupby("production_type")
    .agg(
        rows=("production_hl", "size"),
        regions=("region", "nunique"),
        campaigns=("campaign_year", "nunique"),
        missing_values=("production_hl", lambda x: x.isna().sum()),
    )
    .reset_index()
)

display(summary)


## 4. Convert to machine-learning format

The long dataframe has one row per region, year and production type. For modelling, it is better to have one row per region and year, with one column for each production type.


In [ ]:


production_wide = (
    production_long
    .pivot_table(
        index=["region", "campaign_year", "year_start"],
        columns="production_column",
        values="production_hl",
        aggfunc="first",
    )
    .reset_index()
)

production_wide.columns.name = None

# Ensure all expected columns exist, even if one table was not present in the file.
expected_value_columns = list(TYPE_MAP.values())
for col in expected_value_columns:
    if col not in production_wide.columns:
        production_wide[col] = np.nan

# Reorder columns.
production_wide = production_wide[
    ["region", "campaign_year", "year_start"] + expected_value_columns
]

# Sort chronologically and alphabetically.
production_wide = production_wide.sort_values(["year_start", "region"]).reset_index(drop=True)

display(production_wide.head(20))
print(production_wide.shape)


## 5. Basic quality checks

These checks help confirm whether the cleaned table makes sense before saving it.


In [ ]:

print("Dataset shape:", production_wide.shape)
print("\nColumns:")
print(production_wide.columns.tolist())

print("\nYears:")
print(production_wide["campaign_year"].unique())

print("\nRegions:")
print(production_wide["region"].unique())

duplicates = production_wide.duplicated(subset=["region", "campaign_year"]).sum()
print("\nDuplicated region-year rows:", duplicates)

missing = production_wide.isna().sum().sort_values(ascending=False)
print("\nMissing values:")
display(missing.to_frame("missing_count"))

# The target variable for the regression project.
target_col = "total_production_hl"

if target_col in production_wide.columns:
    missing_target = production_wide[target_col].isna().sum()
    print(f"\nMissing target values in {target_col}:", missing_target)

    display(production_wide.describe(include="all"))
else:
    print(f"Warning: target column {target_col} was not found.")


## 6. Save processed data

The cleaned CSV is saved inside `data/processed`.


In [ ]:

production_wide.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print("Saved processed dataset to:")
print(OUTPUT_FILE)

# Reload the saved file to confirm it was written correctly.
check = pd.read_csv(OUTPUT_FILE)
display(check.head())
print(check.shape)


## 7. Optional quick modelling check

This is not the final modelling section. It is only a quick check to confirm that the cleaned dataset can be used in a scikit-learn pipeline.


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_df = production_wide.dropna(subset=["total_production_hl"]).copy()

feature_cols = [
    "region",
    "year_start",
    "dop_production_hl",
    "igp_production_hl",
    "year_variety_production_hl",
    "non_certified_production_hl",
]

# Keep only available features.
feature_cols = [col for col in feature_cols if col in model_df.columns]

X = model_df[feature_cols]
y = model_df["total_production_hl"]

# Chronological split: train on older campaigns, test on the most recent campaigns.
# Adjust this if your dataset has fewer or more years.
test_years = sorted(model_df["year_start"].unique())[-3:]

train_mask = ~model_df["year_start"].isin(test_years)
test_mask = model_df["year_start"].isin(test_years)

X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]

numeric_features = [col for col in X.columns if col != "region"]
categorical_features = ["region"] if "region" in X.columns else []

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="drop",
)

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

linear_model.fit(X_train, y_train)
pred = linear_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)

print("Quick modelling check - Linear Regression")
print("Test years:", test_years)
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²:   {r2:.3f}")

results_preview = X_test.copy()
results_preview["actual_total_production_hl"] = y_test
results_preview["predicted_total_production_hl"] = pred
results_preview["absolute_error"] = np.abs(results_preview["actual_total_production_hl"] - results_preview["predicted_total_production_hl"])

display(results_preview.head(20))
